# BGE-M3 Triplet Training with Hugging Face Dataset

This notebook fine-tunes **BAAI/bge-m3** with **triplet loss** using the Sentence-Transformers Trainer API.

It loads the dataset directly from Hugging Face:

`monoboard/thai-land-tax-full-triplets`

Each example has the structure:

```json
{"query": "...", "pos": ["..."], "neg": ["..."]}
```

We will:

1. Load the multi-positive / multi-negative dataset from Hugging Face.
2. Split it into train / validation subsets.
3. Expand each example into (anchor, positive, negative) triplets.
4. Fine-tune **BAAI/bge-m3** using **triplet loss**.


## 0) Prerequisites

```bash
pip install -U sentence-transformers transformers datasets accelerate wandb
```
> If you don't use Weights & Biases, set `report_to="none"` in the training args.


In [ ]:
# 1) Imports & Config
import os, json, torch
from datasets import load_dataset, Dataset
from transformers import EarlyStoppingCallback
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments

# ---- Hugging Face dataset config ----
HF_DATASET_NAME = "monoboard/thai-land-tax-full-triplets"
HF_SPLIT        = "train"  # full dataset stored as a single split

# ---- Output / logging config ----
OUTPUT_DIR = "output_bge_m3_triplet-thai-land-tax"
WANDB_RUN  = "bge-m3-triplet-thai-land-tax"

# ---- Mixed precision helper ----
def bf16_supported():
    if not torch.cuda.is_available():
        return False
    major, minor = torch.cuda.get_device_capability()
    # BF16 broadly supported on Ampere (8.0) and newer
    return (major >= 8)

In [ ]:
# 2) Load Hugging Face dataset & expand to triplets

def expand_to_triplet_dataset(hf_ds):
    """Expand multi-pos/multi-neg examples into anchor/positive/negative triplets."""
    anchors, positives, negatives = [], [], []
    for ex in hf_ds:
        query = ex["query"]
        pos_list = ex["pos"]
        neg_list = ex["neg"]
        if not pos_list or not neg_list:
            continue
        for pos in pos_list:
            for neg in neg_list:
                anchors.append(query)
                positives.append(pos)
                negatives.append(neg)
    return Dataset.from_dict({
        "anchor": anchors,
        "positive": positives,
        "negative": negatives,
    })

# Load the full dataset from Hugging Face
hf_full = load_dataset(HF_DATASET_NAME, split=HF_SPLIT)
print("Full HF dataset size:", len(hf_full))

# Reproducible train/validation split (e.g., 80/20)
split = hf_full.train_test_split(test_size=0.2, seed=42)
hf_train = split["train"]
hf_val   = split["test"]

print("HF train subset:", len(hf_train))
print("HF val subset:", len(hf_val))

# Expand both splits into triplet datasets
train_dataset = expand_to_triplet_dataset(hf_train)
val_dataset   = expand_to_triplet_dataset(hf_val)

print("Triplet train_dataset:", train_dataset)
print("Triplet val_dataset:", val_dataset)

In [ ]:
# 3) Load model & define TripletLoss
device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device_str)

model = SentenceTransformer("BAAI/bge-m3", device=device_str)

triplet_loss = losses.TripletLoss(
    model=model,
    triplet_margin=0.3,
)


In [ ]:
# 4) Training Arguments
training_args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    warmup_steps=500,
    bf16=bf16_supported(),
    logging_steps=1000,
    save_steps=1000,
    save_total_limit=1,
    evaluation_strategy="steps",
    lr_scheduler_type="cosine",
    optim="adamw_torch_fused",
    eval_steps=1000,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",   # set to "none" if not using W&B
    run_name=WANDB_RUN,
)
print(training_args)


In [ ]:
# 5) Initialize Trainer with EarlyStopping
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=triplet_loss,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=10)],
)

print("✅ Trainer ready. Call `trainer.train()` to start fine-tuning.")


In [ ]:
# 6) Train
trainer.train()
